# Anexo — Baseline predictivo honesto (P6)

> **Este notebook es ADICIONAL** al análisis descriptivo principal (Fases 2–4). Su fin es demostrar que los descriptores clásicos **no generalizan** cuando el split respeta la unidad compuesto → puente al proyecto GNN de la JIC.

**NO** reportar el split por filas como resultado válido.


## 0. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from src.paths import PROJECT_ROOT as ROOT, setup_path
setup_path()

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

FIG_DIR = ROOT / "outputs" / "chembl" / "figures"
RESULTS_DIR = ROOT / "outputs" / "chembl" / "results"
for d in (FIG_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

from src.analisis_proyecto.chembl_baseline import (
    honest_baseline_compound_level,
    leaky_baseline_row_level,
)
from src.analisis_proyecto.chembl_preprocessing import load_bioactivity

ACTIVITIES_CSV = ROOT / "data" / "processed" / "activities_clean.csv"
COMPOUNDS_CSV = ROOT / "data" / "processed" / "compounds_features.csv"

activities = load_bioactivity(ACTIVITIES_CSV)
compounds = pd.read_csv(COMPOUNDS_CSV)


## 1. Comparación split compuesto vs split por filas (fuga)

In [ ]:
honest = honest_baseline_compound_level(compounds)
leaky = leaky_baseline_row_level(activities)

metrics = pd.DataFrame([honest, leaky])
display(metrics.round(4))
metrics.to_csv(RESULTS_DIR / "baseline_honest_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(metrics["split"], metrics["r2_test"], color=["#2ca02c", "#d62728"])
ax.set_ylabel("R² test")
ax.set_title("Baseline RF — split honesto vs fuga")
ax.axhline(0, color="gray", lw=0.8)
plt.tight_layout()
plt.savefig(FIG_DIR / "baseline_honest_vs_leaky.png", bbox_inches="tight")
plt.show()


## 2. Conclusión

- **Split por compuesto (honesto):** R² bajo o negativo con 107 compuestos y 8 descriptores.
- **Split por filas (fuga):** R² artificialmente alto — la misma molécula aparece en train y test.
- Con esta señal limitada, los **grafos moleculares (GNN)** son la vía prometedora (proyecto JIC hermano).
